<a href="https://colab.research.google.com/github/pushcodes04/ST422_Team1_Project/blob/main/Dataanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import pandas as pd
import numpy as np

# ─── PATHS ────────────────────────────────────────────────────────────────────
# Update these to match your local folder structure

COLLISION_PATH = "/content/dft-road-casualty-statistics-collision-1979-latest-published-year.csv"
VEHICLE_PATH   = "/content/dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv"
CASUALTY_PATH  = "/content/dft-road-casualty-statistics-casualty-1979-latest-published-year.csv"

# ─── LOAD SETTINGS ────────────────────────────────────────────────────────────
# collision_index must be string to prevent scientific notation corruption
# low_memory=False prevents mixed-type warnings on large files
DTYPE_OVERRIDE = {"collision_index": str, "collision_ref_no": str}

# ─── HELPER FUNCTIONS ─────────────────────────────────────────────────────────

def load_table(path, label):
    """Load a STATS19 CSV and print basic load confirmation."""
    print(f"\n{'='*70}")
    print(f"  Loading: {label}")
    print(f"  Path:    {path}")
    print(f"{'='*70}")
    df = pd.read_csv(path, dtype=DTYPE_OVERRIDE, low_memory=False)
    print(f"  Rows:    {len(df):,}")
    print(f"  Cols:    {len(df.columns)}")
    return df


def explore_table(df, label):
    """Print a full column overview including dtype and missingness."""
    print(f"\n{'='*70}")
    print(f"  COLUMN OVERVIEW — {label}")
    print(f"{'='*70}")

    # Head
    print(f"\n--- First 5 rows ---")
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 30)
    print(df.head().to_string())

    # Column summary
    print(f"\n--- Column summary ({len(df.columns)} columns) ---")
    summary = pd.DataFrame({
        "column":      df.columns,
        "dtype":       df.dtypes.values,
        "n_missing":   df.isnull().sum().values,
        "pct_missing": (df.isnull().sum().values / len(df) * 100).round(1),
        "n_unique":    df.nunique().values,
        "sample_val":  [df[c].dropna().iloc[0] if df[c].dropna().shape[0] > 0
                        else "ALL NULL" for c in df.columns],
    })
    print(summary.to_string(index=False))

    # Flag high missingness
    high_miss = summary[summary["pct_missing"] > 10]
    if len(high_miss) > 0:
        print(f"\n--- Columns with >10% missing ({len(high_miss)}) ---")
        print(high_miss[["column", "pct_missing"]].to_string(index=False))
    else:
        print(f"\n  No columns exceed 10% missingness threshold.")

    # Years present
    if "collision_year" in df.columns:
        years = sorted(df["collision_year"].dropna().unique().astype(int))
        print(f"\n--- Years present ---")
        print(f"  {years[0]} to {years[-1]}  ({len(years)} years)")

    # Severity distribution if present
    if "collision_severity" in df.columns:
        print(f"\n--- collision_severity distribution ---")
        print(df["collision_severity"].value_counts().sort_index().to_string())

    if "casualty_severity" in df.columns:
        print(f"\n--- casualty_severity distribution ---")
        print(df["casualty_severity"].value_counts().sort_index().to_string())


# ─── MAIN ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    print("Loading full STATS19 datasets — this may take 1-2 minutes for large files.")

    # Load all three tables
    collisions = load_table(COLLISION_PATH, "COLLISIONS")
    vehicles   = load_table(VEHICLE_PATH,   "VEHICLES")
    casualties = load_table(CASUALTY_PATH,  "CASUALTIES")

    # Explore each
    explore_table(collisions, "COLLISIONS")
    explore_table(vehicles,   "VEHICLES")
    explore_table(casualties, "CASUALTIES")

    # Cross-table join key check
    print(f"\n{'='*70}")
    print(f"  JOIN KEY CHECK — collision_index")
    print(f"{'='*70}")

    col_idx = set(collisions["collision_index"].dropna().unique())
    veh_idx = set(vehicles["collision_index"].dropna().unique())
    cas_idx = set(casualties["collision_index"].dropna().unique())

    print(f"  Unique collision_index in collisions: {len(col_idx):,}")
    print(f"  Unique collision_index in vehicles:   {len(veh_idx):,}")
    print(f"  Unique collision_index in casualties: {len(cas_idx):,}")

    veh_not_in_col = len(veh_idx - col_idx)
    cas_not_in_col = len(cas_idx - col_idx)
    col_not_in_veh = len(col_idx - veh_idx)
    col_not_in_cas = len(col_idx - cas_idx)

    print(f"\n  Vehicle records with no matching collision:  {veh_not_in_col:,}")
    print(f"  Casualty records with no matching collision: {cas_not_in_col:,}")
    print(f"  Collisions with no matching vehicle:         {col_not_in_veh:,}")
    print(f"  Collisions with no matching casualty:        {col_not_in_cas:,}")

    if veh_not_in_col > 0 or cas_not_in_col > 0:
        print(f"\n  WARNING: Orphaned records detected — review before joining.")
    else:
        print(f"\n  Join key looks clean across all three tables.")

    print(f"\n{'='*70}")
    print(f"  EXPLORATION COMPLETE")
    print(f"  Review output above before proceeding to 01_data_cleaning.py")
    print(f"{'='*70}\n")

Loading full STATS19 datasets — this may take 1-2 minutes for large files.

  Loading: COLLISIONS
  Path:    /content/dft-road-casualty-statistics-collision-1979-latest-published-year.csv
  Rows:    1,902,680
  Cols:    44

  Loading: VEHICLES
  Path:    /content/dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv
  Rows:    3,710,219
  Cols:    32

  Loading: CASUALTIES
  Path:    /content/dft-road-casualty-statistics-casualty-1979-latest-published-year.csv
  Rows:    6,916,615
  Cols:    23

  COLUMN OVERVIEW — COLLISIONS

--- First 5 rows ---
  collision_index  collision_year collision_ref_no  location_easting_osgr  location_northing_osgr  longitude   latitude  police_force  collision_severity  number_of_vehicles  number_of_casualties        date  day_of_week   time  local_authority_district local_authority_ons_district local_authority_highway local_authority_highway_current  first_road_class  first_road_number  road_type  speed_limit  junction_detail_historic  junct

In [ ]:
import pandas as pd
import os

# ─── PATHS ────────────────────────────────────────────────────────────────────
COLLISION_PATH = "/content/dft-road-casualty-statistics-collision-1979-latest-published-year.csv"
VEHICLE_PATH   = "/content/dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv"
CASUALTY_PATH  = "/content/dft-road-casualty-statistics-casualty-1979-latest-published-year.csv"

YEAR_CUTOFF = 2015

# ─── FILTER ───────────────────────────────────────────────────────────────────
# If you already have collisions/vehicles/casualties loaded from the previous
# cell, this just filters them directly without re-reading from disk.
# If not loaded yet, it reads from the paths above.

try:
    collisions
    print("Using already-loaded collisions dataframe")
except NameError:
    print("Loading collisions from disk...")
    collisions = pd.read_csv(COLLISION_PATH, dtype={"collision_index": str, "collision_ref_no": str}, low_memory=False)

try:
    vehicles
    print("Using already-loaded vehicles dataframe")
except NameError:
    print("Loading vehicles from disk...")
    vehicles = pd.read_csv(VEHICLE_PATH, dtype={"collision_index": str, "collision_ref_no": str}, low_memory=False)

try:
    casualties
    print("Using already-loaded casualties dataframe")
except NameError:
    print("Loading casualties from disk...")
    casualties = pd.read_csv(CASUALTY_PATH, dtype={"collision_index": str, "collision_ref_no": str}, low_memory=False)

# ─── APPLY YEAR FILTER ────────────────────────────────────────────────────────
print(f"\nFiltering to {YEAR_CUTOFF} onwards...")

col_before = len(collisions)
veh_before = len(vehicles)
cas_before = len(casualties)

collisions = collisions[collisions["collision_year"] >= YEAR_CUTOFF].copy().reset_index(drop=True)
vehicles   = vehicles[vehicles["collision_year"]   >= YEAR_CUTOFF].copy().reset_index(drop=True)
casualties = casualties[casualties["collision_year"] >= YEAR_CUTOFF].copy().reset_index(drop=True)

print(f"\n  collisions: {col_before:,} → {len(collisions):,} rows  ({col_before - len(collisions):,} dropped)")
print(f"  vehicles:   {veh_before:,} → {len(vehicles):,} rows  ({veh_before - len(vehicles):,} dropped)")
print(f"  casualties: {cas_before:,} → {len(casualties):,} rows  ({cas_before - len(casualties):,} dropped)")

# ─── YEAR RANGE CONFIRMATION ──────────────────────────────────────────────────
print(f"\n  collisions years: {int(collisions['collision_year'].min())} – {int(collisions['collision_year'].max())}")
print(f"  vehicles   years: {int(vehicles['collision_year'].min())} – {int(vehicles['collision_year'].max())}")
print(f"  casualties years: {int(casualties['collision_year'].min())} – {int(casualties['collision_year'].max())}")

# ─── JOIN KEY CHECK ───────────────────────────────────────────────────────────
print(f"\n--- Post-filter join key check ---")
col_idx = set(collisions["collision_index"].dropna().unique())
veh_idx = set(vehicles["collision_index"].dropna().unique())
cas_idx = set(casualties["collision_index"].dropna().unique())

print(f"  Unique collision_index in collisions: {len(col_idx):,}")
print(f"  Unique collision_index in vehicles:   {len(veh_idx):,}")
print(f"  Unique collision_index in casualties: {len(cas_idx):,}")
print(f"  Vehicle orphans  (no matching collision): {len(veh_idx - col_idx):,}")
print(f"  Casualty orphans (no matching collision): {len(cas_idx - col_idx):,}")
print(f"  Collisions with no vehicle:  {len(col_idx - veh_idx):,}")
print(f"  Collisions with no casualty: {len(col_idx - cas_idx):,}")

# ─── KEY MISSINGNESS CHECK ────────────────────────────────────────────────────
print(f"\n--- Missingness on key columns after filter ---")

lat_pct = collisions["latitude"].isnull().mean() * 100
lon_pct = collisions["longitude"].isnull().mean() * 100
adj_pct = collisions["collision_adjusted_severity_serious"].isnull().mean() * 100

print(f"  latitude  missing:                           {lat_pct:.1f}%")
print(f"  longitude missing:                           {lon_pct:.1f}%")
print(f"  collision_adjusted_severity_serious missing: {adj_pct:.1f}%")

# ─── SEVERITY DISTRIBUTION ────────────────────────────────────────────────────
print(f"\n--- collision_severity distribution (2015+) ---")
sev_map = {1: "Fatal", 2: "Serious", 3: "Slight"}
print(collisions["collision_severity"].map(sev_map).value_counts().to_string())

print(f"\n--- casualty_severity distribution (2015+) ---")
print(casualties["casualty_severity"].map(sev_map).value_counts().to_string())

print(f"\nDone. collisions, vehicles, casualties now filtered to {YEAR_CUTOFF}+.")
print("Ready for cleaning.")

Using already-loaded collisions dataframe
Using already-loaded vehicles dataframe
Using already-loaded casualties dataframe

Filtering to 2015 onwards...

  collisions: 1,902,680 → 244,104 rows  (1,658,576 dropped)
  vehicles:   3,710,219 → 535,856 rows  (3,174,363 dropped)
  casualties: 6,916,615 → 846,305 rows  (6,070,310 dropped)

  collisions years: 2015 – 2024
  vehicles   years: 2015 – 2024
  casualties years: 2015 – 2024

--- Post-filter join key check ---
  Unique collision_index in collisions: 244,104
  Unique collision_index in vehicles:   465,147
  Unique collision_index in casualties: 717,932
  Vehicle orphans  (no matching collision): 366,297
  Casualty orphans (no matching collision): 565,620
  Collisions with no vehicle:  145,254
  Collisions with no casualty: 91,792

--- Missingness on key columns after filter ---
  latitude  missing:                           0.0%
  longitude missing:                           0.0%
  collision_adjusted_severity_serious missing: 0.0%

-

In [4]:
collisions.to_csv("/content/collisions_2015_plus.csv", index=False)
vehicles.to_csv("/content/vehicles_2015_plus.csv",     index=False)
casualties.to_csv("/content/casualties_2015_plus.csv", index=False)

print("Saved:")
print(f"  /content/collisions_2015_plus.csv  —  {len(collisions):,} rows")
print(f"  /content/vehicles_2015_plus.csv    —  {len(vehicles):,} rows")
print(f"  /content/casualties_2015_plus.csv  —  {len(casualties):,} rows")

Saved:
  /content/collisions_2015_plus.csv  —  244,104 rows
  /content/vehicles_2015_plus.csv    —  535,856 rows
  /content/casualties_2015_plus.csv  —  846,305 rows


In [5]:
# Diagnose join key mismatch
print("--- Sample collision_index values ---")
print("\ncollisions:")
print(collisions["collision_index"].head(10).tolist())

print("\nvehicles:")
print(vehicles["collision_index"].head(10).tolist())

print("\ncasualties:")
print(casualties["collision_index"].head(10).tolist())

# Check key lengths
print("\n--- collision_index string lengths ---")
print("collisions:", collisions["collision_index"].str.len().value_counts().head(5).to_string())
print("\nvehicles:",   vehicles["collision_index"].str.len().value_counts().head(5).to_string())
print("\ncasualties:", casualties["collision_index"].str.len().value_counts().head(5).to_string())

# Check overlap directly
col_idx = set(collisions["collision_index"].dropna().unique())
veh_idx = set(vehicles["collision_index"].dropna().unique())
cas_idx = set(casualties["collision_index"].dropna().unique())

# Sample some orphaned vehicle keys
veh_orphans = list(veh_idx - col_idx)[:5]
print(f"\n--- Sample orphaned vehicle collision_index values ---")
print(veh_orphans)

# Sample some orphaned casualty keys
cas_orphans = list(cas_idx - col_idx)[:5]
print(f"\n--- Sample orphaned casualty collision_index values ---")
print(cas_orphans)

# Check if orphaned keys exist in collisions under a different format
print(f"\n--- Do orphaned vehicle keys appear in collisions at all? ---")
for key in veh_orphans:
    match = collisions[collisions["collision_index"].str.contains(key[-6:], na=False)]
    print(f"  {key} → {len(match)} partial matches in collisions")

--- Sample collision_index values ---

collisions:
['2018450290294', '2020230984311', '2019010177739', '2022991145836', '2019470863182', '2019410836238', '2021471033730', '201897GP70612', '2023010424647', '2016621601060']

vehicles:
['2015542933515', '201501RY10180', '2015405DA0593', '201520M013905', '2015230039137', '20154100G0487', '2015160A02971', '2015450026503', '2015300015880', '201506P002357']

casualties:
['201523N026534', '2015460259328', '201501GD10100', '2015621500089', '20151324M1158', '201501WW50934', '2015200032984', '201501TX20978', '20154100F0695', '2015051500506']

--- collision_index string lengths ---
collisions: collision_index
13    244104

vehicles: collision_index
13    535856

casualties: collision_index
13    846305

--- Sample orphaned vehicle collision_index values ---
['201522E503723', '2022991162010', '2019521905627', '2019040852625', '2016360121007']

--- Sample orphaned casualty collision_index values ---
['201522E503723', '2022991162010', '2019521905627'

In [6]:
# Quantify orphan rate by year
col_idx = set(collisions["collision_index"].dropna().unique())

# Tag orphans in vehicles and casualties
vehicles["has_collision"]   = vehicles["collision_index"].isin(col_idx)
casualties["has_collision"] = casualties["collision_index"].isin(col_idx)

print("--- Vehicle orphan rate by year ---")
veh_by_year = vehicles.groupby("collision_year")["has_collision"].agg(
    total="count",
    matched="sum"
)
veh_by_year["orphan_pct"] = ((veh_by_year["total"] - veh_by_year["matched"]) / veh_by_year["total"] * 100).round(1)
print(veh_by_year.to_string())

print("\n--- Casualty orphan rate by year ---")
cas_by_year = casualties.groupby("collision_year")["has_collision"].agg(
    total="count",
    matched="sum"
)
cas_by_year["orphan_pct"] = ((cas_by_year["total"] - cas_by_year["matched"]) / cas_by_year["total"] * 100).round(1)
print(cas_by_year.to_string())

--- Vehicle orphan rate by year ---
                total  matched  orphan_pct
collision_year                            
2015.0          60461    12743        78.9
2016.0          63339    13456        78.8
2017.0          56168    12028        78.6
2018.0          60640    12844        78.8
2019.0          55161    11647        78.9
2020.0          40086     8554        78.7
2021.0          49099    10378        78.9
2022.0          51223    10938        78.6
2023.0          49326    10451        78.8
2024.0          50353    10721        78.7

--- Casualty orphan rate by year ---
                 total  matched  orphan_pct
collision_year                             
2015            105835    22307        78.9
2016            102821    21634        79.0
2017             97311    20789        78.6
2018             90641    19428        78.6
2019             86624    18459        78.7
2020             65784    13780        79.1
2021             72717    15280        79.0
2022          